In [ ]:
import argparse
import os
import random

import numpy as np
import torch
import torchvision.models as models
from torchvision import transforms

from attack_utils import run_experiment
from impl_apgd_dlr import APGD_DLR_Linf

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

datasets = {
    'correct_1000_AlexNet': ('alexnet', models.AlexNet_Weights.IMAGENET1K_V1),
    'correct_1000_DenseNet121': ('densenet121', models.DenseNet121_Weights.IMAGENET1K_V1),
    'correct_1000_GoogLeNet': ('googlenet', models.GoogLeNet_Weights.IMAGENET1K_V1),
    'correct_1000_MobileNetV3_large': ('mobilenet_v3_large', models.MobileNet_V3_Large_Weights.IMAGENET1K_V1),
    'correct_1000_ResNet34': ('resnet34', models.ResNet34_Weights.IMAGENET1K_V1),
    'correct_1000_VGG11': ('vgg11', models.VGG11_Weights.IMAGENET1K_V1),
    'correct_1000_EfficientNet_b0': ('efficientnet_b0', models.EfficientNet_B0_Weights.IMAGENET1K_V1),
}

config = {
    'SEED': 42,
    'selected_count': 500,
    'output_dir': 'adversarial_samples',
    'attack_name': 'apgd_dlr',
    'MAX_SAVE_ADV': 10,
    'if_save_adv': False,
    'threshold': 1e-6,
    'target_labels': torch.tensor([100]).to(device),
    'if_target': False,
    'if_prune': False,
    'eps': 100/255,
    'alpha': 1/255,
    'steps': 100,
    'random_start': False,
    'targeted': False,
    'n_restarts': 2,
    'rho': 0.75,
    'eot_iter': 1,
    'early_stop': True,
    'step_ratio': 0.01,
    'max_ratio': 1.0,
}

parser = argparse.ArgumentParser(description='Adversarial Attack Experiment')
parser.add_argument('--eps', type=float, default=config['eps'], help='Epsilon for APGD-DLR')
parser.add_argument('--alpha', type=float, default=config['alpha'], help='Initial step size for APGD-DLR')
parser.add_argument('--steps', type=int, default=config['steps'], help='Steps for APGD-DLR')
parser.add_argument('--n_restarts', type=int, default=config['n_restarts'], help='Restarts for APGD-DLR')
parser.add_argument('--rho', type=float, default=config['rho'], help='Oscillation threshold for APGD-DLR')
parser.add_argument('--eot_iter', type=int, default=config['eot_iter'], help='EOT iterations for APGD-DLR')
args = parser.parse_args([])

config.update(vars(args))

SEED = config['SEED']
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


使用设备: cuda


In [2]:
if __name__ == "__main__":
    for dataset_name, (model_name, weights) in datasets.items():
        print(f"\n=== 开始实验: {dataset_name} 使用模型 {model_name} ===")

        model = getattr(models, model_name)(weights=weights).to(device)
        model.eval()

        transform = transforms.Compose([
            transforms.Resize(256),
            transforms.ToTensor(),
        ])

        config['val_dir'] = f"./{dataset_name}"
        config['attack_name'] = f"apgd_dlr_{model_name}"

        val_dir = config['val_dir']
        all_images = [f for f in os.listdir(val_dir) if f.endswith('.JPEG')]
        selected_images = random.sample(all_images, min(config['selected_count'], len(all_images)))

        output_dir = config['output_dir']
        attack_name = config['attack_name']
        attack_output_dir = os.path.join(output_dir, attack_name)
        os.makedirs(attack_output_dir, exist_ok=True)

        results = run_experiment(
            config=config,
            transform=transform,
            model=model,
            device=device,
            selected_images=selected_images,
            attack_output_dir=attack_output_dir,
            attack_cls=APGD_DLR_Linf,
        )

        print(f"=== 实验完成: {dataset_name} ===")



=== 开始实验: correct_1000_AlexNet 使用模型 alexnet ===


apgd_dlr_alexnet: 100%|██████████| 500/500 [01:22<00:00,  6.05it/s]


配置: {'SEED': 42, 'selected_count': 500, 'output_dir': 'adversarial_samples', 'attack_name': 'apgd_dlr_alexnet', 'MAX_SAVE_ADV': 10, 'if_save_adv': False, 'threshold': 1e-06, 'target_labels': tensor([100], device='cuda:0'), 'if_target': True, 'if_prune': False, 'eps': 0.39215686274509803, 'alpha': 0.00392156862745098, 'steps': 100, 'random_start': False, 'targeted': True, 'n_restarts': 2, 'rho': 0.75, 'eot_iter': 1, 'early_stop': True, 'step_ratio': 0.01, 'max_ratio': 1.0, 'val_dir': './correct_1000_AlexNet'}
成功攻击: 441/500
攻击成功率: 88.20%
平均被修改像素点数量: 85141.90
平均被修改像素比例: 98.18%
平均扰动均值: 11832.79
平均 SSIM: 0.930234
平均 PSNR: 36.9109dB
SSIM >= 0.975: 14.74%
SSIM >= 0.98: 9.07%
SSIM >= 0.985: 5.44%
SSIM >= 0.99: 1.13%
SSIM >= 0.995: 0.23%
PSNR >= 39dB: 16.55%
PSNR >= 41dB: 4.08%
PSNR >= 43dB: 0.91%
PSNR >= 45dB: 0.45%
PSNR >= 47dB: 0.23%
=== 实验完成: correct_1000_AlexNet ===

=== 开始实验: correct_1000_DenseNet121 使用模型 densenet121 ===


apgd_dlr_densenet121: 100%|██████████| 500/500 [05:08<00:00,  1.62it/s]


配置: {'SEED': 42, 'selected_count': 500, 'output_dir': 'adversarial_samples', 'attack_name': 'apgd_dlr_densenet121', 'MAX_SAVE_ADV': 10, 'if_save_adv': False, 'threshold': 1e-06, 'target_labels': tensor([100], device='cuda:0'), 'if_target': True, 'if_prune': False, 'eps': 0.39215686274509803, 'alpha': 0.00392156862745098, 'steps': 100, 'random_start': False, 'targeted': True, 'n_restarts': 2, 'rho': 0.75, 'eot_iter': 1, 'early_stop': True, 'step_ratio': 0.01, 'max_ratio': 1.0, 'val_dir': './correct_1000_DenseNet121'}
成功攻击: 498/500
攻击成功率: 99.60%
平均被修改像素点数量: 83622.09
平均被修改像素比例: 97.08%
平均扰动均值: 7756.91
平均 SSIM: 0.966091
平均 PSNR: 40.3774dB
SSIM >= 0.975: 35.14%
SSIM >= 0.98: 26.71%
SSIM >= 0.985: 17.07%
SSIM >= 0.99: 7.63%
SSIM >= 0.995: 1.41%
PSNR >= 39dB: 83.13%
PSNR >= 41dB: 30.52%
PSNR >= 43dB: 5.22%
PSNR >= 45dB: 1.00%
PSNR >= 47dB: 0.20%
=== 实验完成: correct_1000_DenseNet121 ===

=== 开始实验: correct_1000_GoogLeNet 使用模型 googlenet ===


...

配置: {'SEED': 42, 'selected_count': 500, 'output_dir': 'adversarial_samples', 'attack_name': 'apgd_dlr_resnet34', 'MAX_SAVE_ADV': 10, 'if_save_adv': False, 'threshold': 1e-06, 'target_labels': tensor([100], device='cuda:0'), 'if_target': True, 'if_prune': False, 'eps': 0.39215686274509803, 'alpha': 0.00392156862745098, 'steps': 100, 'random_start': False, 'targeted': True, 'n_restarts': 2, 'rho': 0.75, 'eot_iter': 1, 'early_stop': True, 'step_ratio': 0.01, 'max_ratio': 1.0, 'val_dir': './correct_1000_ResNet34'}
成功攻击: 492/500
攻击成功率: 98.40%
平均被修改像素点数量: 84074.86
平均被修改像素比例: 96.97%
平均扰动均值: 7959.41
平均 SSIM: 0.964261
平均 PSNR: 40.2453dB
SSIM >= 0.975: 33.94%
SSIM >= 0.98: 27.24%
SSIM >= 0.985: 17.28%
SSIM >= 0.99: 6.71%
SSIM >= 0.995: 2.03%
PSNR >= 39dB: 77.85%
PSNR >= 41dB: 29.88%
PSNR >= 43dB: 6.30%
PSNR >= 45dB: 0.81%
PSNR >= 47dB: 0.41%
=== 实验完成: correct_1000_ResNet34 ===

=== 开始实验: correct_1000_VGG11 使用模型 vgg11 ===


apgd_dlr_vgg11: 100%|██████████| 500/500 [04:14<00:00,  1.97it/s]


配置: {'SEED': 42, 'selected_count': 500, 'output_dir': 'adversarial_samples', 'attack_name': 'apgd_dlr_vgg11', 'MAX_SAVE_ADV': 10, 'if_save_adv': False, 'threshold': 1e-06, 'target_labels': tensor([100], device='cuda:0'), 'if_target': True, 'if_prune': False, 'eps': 0.39215686274509803, 'alpha': 0.00392156862745098, 'steps': 100, 'random_start': False, 'targeted': True, 'n_restarts': 2, 'rho': 0.75, 'eot_iter': 1, 'early_stop': True, 'step_ratio': 0.01, 'max_ratio': 1.0, 'val_dir': './correct_1000_VGG11'}
成功攻击: 465/500
攻击成功率: 93.00%
平均被修改像素点数量: 84025.13
平均被修改像素比例: 97.68%
平均扰动均值: 9140.63
平均 SSIM: 0.951417
平均 PSNR: 39.0987dB
SSIM >= 0.975: 26.88%
SSIM >= 0.98: 19.14%
SSIM >= 0.985: 11.83%
SSIM >= 0.99: 6.24%
SSIM >= 0.995: 0.86%
PSNR >= 39dB: 48.82%
PSNR >= 41dB: 16.77%
PSNR >= 43dB: 4.09%
PSNR >= 45dB: 1.08%
PSNR >= 47dB: 0.43%
=== 实验完成: correct_1000_VGG11 ===

=== 开始实验: correct_1000_EfficientNet_b0 使用模型 efficientnet_b0 ===


apgd_dlr_efficientnet_b0: 100%|██████████| 500/500 [04:42<00:00,  1.77it/s]

配置: {'SEED': 42, 'selected_count': 500, 'output_dir': 'adversarial_samples', 'attack_name': 'apgd_dlr_efficientnet_b0', 'MAX_SAVE_ADV': 10, 'if_save_adv': False, 'threshold': 1e-06, 'target_labels': tensor([100], device='cuda:0'), 'if_target': True, 'if_prune': False, 'eps': 0.39215686274509803, 'alpha': 0.00392156862745098, 'steps': 100, 'random_start': False, 'targeted': True, 'n_restarts': 2, 'rho': 0.75, 'eot_iter': 1, 'early_stop': True, 'step_ratio': 0.01, 'max_ratio': 1.0, 'val_dir': './correct_1000_EfficientNet_b0'}
成功攻击: 476/500
攻击成功率: 95.20%
平均被修改像素点数量: 85031.51
平均被修改像素比例: 98.04%
平均扰动均值: 11313.06
平均 SSIM: 0.933609
平均 PSNR: 37.2271dB
SSIM >= 0.975: 14.92%
SSIM >= 0.98: 9.66%
SSIM >= 0.985: 5.88%
SSIM >= 0.99: 1.68%
SSIM >= 0.995: 0.00%
PSNR >= 39dB: 18.70%
PSNR >= 41dB: 5.04%
PSNR >= 43dB: 0.84%
PSNR >= 45dB: 0.00%
PSNR >= 47dB: 0.00%
=== 实验完成: correct_1000_EfficientNet_b0 ===
